# Initial setup

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
import sys


PROJECT_ROOT = Path.cwd().parent
print(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))
    print(f"Added {PROJECT_ROOT} to path")

/home/henrik_arch/Coding/macrodata-norway
Added /home/henrik_arch/Coding/macrodata-norway to path


# Data exploration

In [16]:
from src.sources.ssb import get_available_tables

In [17]:
tables = get_available_tables(query ='title:Konsumprisindeks', pages = 2)

In [22]:
print(tables[0])

['14700: Konsumprisindeks (KPI), etter vare- og tjenestegruppe (2025=100) 2000M01-2026M04', '14700', '2026-05-11T06:00:00Z']


In [ ]:

import requests
import itertools
import pandas as pd

response = requests.get(
    "https://data.ssb.no/api/pxwebapi/v2/tables/14706/data"
)

# the whole dataset as JSON-stat2
js2_data = response.json()

# Get the dimensions of the dataset
dimensions = {
    dim_name: list(dim_info["category"]["index"].keys())
    for dim_name, dim_info in js2_data["dimension"].items()
}
# Get the code to the dataframe
dimension_combinations = list(itertools.product(*list(dimensions.values())))
df = pd.DataFrame(dimension_combinations, columns=dimensions.keys())

# Get the textlabels and add them next to codecolumn
dimensions_labels = {
    dim_name: dim_info["category"]["label"]
    for dim_name, dim_info in js2_data["dimension"].items()
}
for i, (col, codelist) in enumerate(dimensions_labels.items()):
    label_colname = f"{col}_label"
    df.insert((i * 2 + 1), label_colname, df[col].map(codelist))

# Add values to the dataframe
df["Value"] = js2_data["value"]

# Get metadata from dataset
metadata = dict(
    table_id=js2_data["extension"]["px"]["tableid"],
    short_title=js2_data["extension"]["px"]["contents"],
    title=js2_data["label"],
    source=js2_data["source"],
    last_update=js2_data["updated"],
    # include footnote, needs post processing
    # note = js2_data['note']
)

In [3]:
from henrik_db import get_connection

with get_connection() as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT datname
            FROM pg_database
            WHERE datistemplate = false
            ORDER BY datname;
        """)
        databases = cur.fetchall()

print(databases)

[('finance_data',), ('macrodata-norway',), ('postgres',)]


In [4]:
from henrik_db import get_connection

with get_connection() as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT table_schema, table_name
            FROM information_schema.tables
            WHERE table_schema NOT IN ('pg_catalog', 'information_schema')
            ORDER BY table_schema, table_name;
        """)
        tables = cur.fetchall()

for schema, table in tables:
    print(f"{schema}.{table}")

In [28]:
df

,KPIavledetSerie,KPIavledetSerie_label,Tid,Tid_label,ContentsCode,ContentsCode_label,Value
0,KPI,Konsumprisindeksen totalt (KPI),2026M04,2026M04,KPIJustIndMnd,Indeks (2025=100),102.8
1,KPI,Konsumprisindeksen totalt (KPI),2026M04,2026M04,Manedsendring,Månedsendring (prosent),0.4
2,KPI,Konsumprisindeksen totalt (KPI),2026M04,2026M04,Tolvmanedersendring,12-måneders endring (prosent),3.4
3,KPI-JE,KPI uten energivarer (KPI-JE),2026M04,2026M04,KPIJustIndMnd,Indeks (2025=100),102.8
4,KPI-JE,KPI uten energivarer (KPI-JE),2026M04,2026M04,Manedsendring,Månedsendring (prosent),0.6
5,KPI-JE,KPI uten energivarer (KPI-JE),2026M04,2026M04,Tolvmanedersendring,12-måneders endring (prosent),3.3
6,KPI-JEL,KPI uten elektrisitet (KPI-JEL),2026M04,2026M04,KPIJustIndMnd,Indeks (2025=100),102.8
7,KPI-JEL,KPI uten elektrisitet (KPI-JEL),2026M04,2026M04,Manedsendring,Månedsendring (prosent),0.4
8,KPI-JEL,KPI uten elektrisitet (KPI-JEL),2026M04,2026M04,Tolvmanedersendring,12-måneders endring (prosent),3.4
9,KPI-JA,KPI justert for avgiftsendringer (KPI-JA),2026M04,2026M04,KPIJustIndMnd,Indeks (2025=100),104.0
